# Senate Profile LLM Extraction Pipeline
**DSBA 6010 — Chloe Partridge**

Aligned with Liu et al. (USENIX Security 2025) *Evaluating LLM-based Personal Information Extraction and Countermeasures*.

Key features:
- **Prompt-style ablation** — direct, pseudocode, ICL (Section 4.2 / Table 13)
- **Religion signal annotation** — LLM-based pre-classification of bio text as `explicit` / `not_explicit` (Section 8b)
- **Traditional baselines** — keyword search, regex, spaCy NER (Tables 4–5)
- **Evaluation metrics** — Accuracy, Rouge-1, BERT score with stratified religion analysis (Section 6.1.4)
- **Model comparison** — 8B vs 70B (Table 3 / Section 6.2)

In [1]:
# Install required packages for all supported providers
# !pip install beautifulsoup4 google-generativeai openai groq pandas tqdm rouge-score bert-score spacy
# !python -m spacy download en_core_web_sm

## Installation & Dependencies

Install required packages for LLM API providers and NLP processing via spaCy.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import json, time, re, random, os, datetime
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

# SESSION INITIALIZATION
# Initialize the entire pipeline session with one factory call.
# The modules.session_init module provides initialize_pipeline_session(),
# which replaces ~80 lines of manual configuration setup.
#
# What initialize_pipeline_session() does:
#   1. Loads model configuration from JSON (groq_config_extraction.json)
#   2. Sets up API client (Groq with GROQ_API_KEY env var or config file)
#   3. Initializes spaCy NER model (en_core_web_sm)
#   4. Collects HTML files from external_data/senate_html/
#   5. Creates PipelineConfig dataclass with all settings
#
# Returns a dictionary with all session state ready to use: session_config, 
# html_files, nlp, output_dir, html_dir.

from modules import initialize_pipeline_session

# Initialize the session (loads config, API client, spaCy model, HTML files)
session = initialize_pipeline_session(
    config_path="../configs/model_configs/groq_config_extraction.json",
    prompt_styles=["direct", "pseudocode", "icl"]
)

# Unpack session for convenient access
session_config = session['session_config']
html_files = session['html_files']
nlp = session['nlp']
OUTPUT_DIR = session['output_dir']
HTML_DIR = session['html_dir']

# Initialize ablation subset with fixed seed for reproducibility
# Fixed seed=42 ensures same 25 senators across runs for reproducible ablation study
import random as py_random
py_random.seed(42)

# Select 25 senators for ablation study
ablation_senators = py_random.sample(
    [f.stem for f in html_files],
    min(25, len(html_files))
)

print(f"✓ Model: {session_config.model} | {len(html_files)} HTML files")
print(f"✓ Prompt styles: {', '.join(session_config.prompt_styles)}")
print(f"✓ Output dir: {OUTPUT_DIR}")
print(f"✓ Ablation subset: {len(ablation_senators)} senators (seed=42)")

/Users/chloe/LLM-Based-Personal-Profile-Extraction/pie_env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✓ Model: llama-3.1-8b-instant | 100 HTML files
✓ Prompt styles: direct, pseudocode, icl
✓ Output dir: /Users/chloe/LLM-Based-Personal-Profile-Extraction/experiments/../outputs/senate_results
✓ Ablation subset: 25 senators (seed=42)


## API Infrastructure

Helper functions for LLM API calls with automatic retry on rate-limit errors.
Includes exponential backoff, JSON parsing, and model-agnostic abstraction layer.

In [4]:
# LLM-BASED EXTRACTION — Task 1 Profile Extraction
# Extract full name, birthdate, gender, education, committee roles, religious affiliation
# from Senate HTML profiles using Groq API with configurable prompt engineering strategies.
#
# This section uses modules.pipeline_runner.run_main_pipeline() which provides:
#   - Resume-safe extraction (skips already-processed senators)
#   - Multi-style prompt testing (direct, pseudocode, ICL)
#   - Rate limiting and automatic retry on quota errors
#   - Incremental JSON save (results_raw.json)
#
# Approx runtime: 2-5 minutes depending on model + number of styles

from modules import run_main_pipeline

# Execute the main pipeline with resume capability
# Returns: dict with "results", "output_path", "done_count", "total_count"
pipeline_result = run_main_pipeline(
    html_files=html_files,
    session_config=session_config,
    resume=True  # Skip already-processed senators
)

print(f"\n✓ Pipeline complete: {pipeline_result['done_count']}/{pipeline_result['total_count']} senators processed")

✓ Resuming: 100 senators already processed
Senators remaining: 0/100
Rate limit: 6s between calls
Styles: direct, pseudocode, icl



Processing senators: 0it [00:00, ?it/s]


✓ Pipeline complete: 100 senators processed

✓ Pipeline complete: 100/100 senators processed


In [5]:
# CSV FORMATTING & DATA PREPARATION
# Convert raw JSON results to CSV format for easier analysis.
# This transforms the nested JSON output from run_main_pipeline into a flat pandas DataFrame.
# Uses modules.config_unified.T1_FIELDS to ensure consistent field ordering.

from modules import T1_FIELDS

with open(OUTPUT_DIR / "results_raw.json") as f:
    raw_results = json.load(f)

# Flatten Task 1 results to CSV
task1_rows = []
for r in raw_results:
    t1_data = r.get("task1_pii", {})
    prompt_style = r.get("prompt_style", "unknown")
    
    # Handle both single-style and all-styles results
    if prompt_style == "all_styles":
        # All three styles were run — create separate rows for each style
        for style_name, style_result in t1_data.items():
            row = {
                "senator_id": r["senator_id"],
                "prompt_style": style_name,
                "extraction_error": style_result.get("error")
            }
            for field in T1_FIELDS:
                row[field] = style_result.get(field)
            task1_rows.append(row)
    else:
        # Single style was run
        row = {
            "senator_id": r["senator_id"],
            "prompt_style": prompt_style,
            "extraction_error": t1_data.get("error")
        }
        for field in T1_FIELDS:
            row[field] = t1_data.get(field)
        task1_rows.append(row)

df_pred = pd.DataFrame(task1_rows)
df_pred.to_csv(OUTPUT_DIR / "task1_pii.csv", index=False)
print(f"✓ Saved {len(df_pred)} rows to task1_pii.csv")
print(f"\nPrompt styles processed:")
print(df_pred["prompt_style"].value_counts().to_string())

✓ Saved 300 rows to task1_pii.csv

Prompt styles processed:
prompt_style
direct        100
pseudocode    100
icl           100


In [6]:
# FIELD COVERAGE ANALYSIS
# Display extraction coverage statistics across all fields in the LLM predictions.
# This shows what percentage of records have non-null values for each extracted field.

print("Field Coverage Statistics (LLM Extraction Results)")
print("=" * 60)
for col in ["full_name", "birthdate", "gender", "education", "committee_roles", "religious_affiliation"]:
    if col in df_pred.columns:
        filled = df_pred[col].notna().sum()
        pct = 100 * filled / len(df_pred)
        print(f"  {col:30s}: {filled:3d}/{len(df_pred)} ({pct:5.1f}%)")

Field Coverage Statistics (LLM Extraction Results)
  full_name                     : 299/300 ( 99.7%)
  birthdate                     :  49/300 ( 16.3%)
  gender                        : 221/300 ( 73.7%)
  education                     : 281/300 ( 93.7%)
  committee_roles               : 285/300 ( 95.0%)
  religious_affiliation         :  85/300 ( 28.3%)


## Baseline Comparison Methods

Implements traditional information extraction methods for comparison with LLM results (Liu et al. Section 5, Tables 4–5):
- **Regex baseline** — Rule-based patterns for name, email, phone
- **spaCy NER baseline** — Pre-trained neural entity recognition for PERSON, ORG, GEO

**Purpose:** Quantify LLM superiority on canonical PII extraction tasks. 
This section tests both methods on the same HTML data then compares coverage vs. LLM accuracy.

In [7]:
# BASELINE EXTRACTION SETUP
# Run traditional extraction methods (regex & spaCy NER) on the same HTML data.
# This allows direct comparison of extraction coverage: traditional methods vs LLM.
#
# Approx runtime: 30-60 seconds (fast — no API calls)

In [8]:
# EXECUTE BASELINE EXTRACTORS
# Uses modules.pipeline_runner.run_baselines() which orchestrates:
#   - Regex baseline (hand-crafted name/email/phone patterns)
#   - spaCy NER baseline (pre-trained PERSON, ORG, GEO entity recognition)
# Returns coverage statistics and saves results to baselines.csv

from modules import run_baselines, REGEX_PATTERNS

baseline_metrics = run_baselines(
    html_files=html_files,
    nlp=nlp,
    regex_patterns=REGEX_PATTERNS,
    output_dir=OUTPUT_DIR
)

print(f"✓ Baseline extraction complete")
print(f"  Results saved to: {OUTPUT_DIR / 'baselines.csv'}")


Running baselines on 100 profiles...


Baselines:   0%|          | 0/100 [00:00<?, ?it/s]

✓ Saved baseline results to /Users/chloe/LLM-Based-Personal-Profile-Extraction/experiments/../outputs/senate_results/baselines.csv

✓ Baseline extraction complete
  Results saved to: /Users/chloe/LLM-Based-Personal-Profile-Extraction/experiments/../outputs/senate_results/baselines.csv


## Ground Truth Extraction & Building

(Re)generate ground truth CSV from Wikipedia, Ballotpedia, and Pew religion data.

This step is optional if ground_truth CSV already exists and is up-to-date. 
Use this to refresh ground truth with new scraper versions or updated senator data.

**Key Features:**
- Resume-safe: Skips senators already processed; continue from checkpoint
- Rate-limited: Respectful delays between requests to external sites
- Merged sources: Wikipedia (bio data) + Ballotpedia (committees) + Pew (religion)
- Logging: Warnings logged for any scrape failures; check if needed

**Runtime:** ~1-2 minutes for full senate (104 senators × 0.5s delay) or faster with resume

In [3]:
# Import ground truth builder
import sys
sys.path.insert(0, '../scripts')
from build_ground_truth import build_ground_truth

# Define paths
GT_HTML_DIR = Path("../external_data/senate_html")
GT_PEW_PATH = Path("../external_data/ground_truth/pew_religion.csv")
GT_OUTPUT_PATH = Path("../external_data/ground_truth/senate_ground_truth.csv")
GT_COMMITTEE_YAML_PATH = Path("../external_data/ground_truth/committee-membership-current.yaml")
GT_COMMITTEES_YAML_PATH = Path("../external_data/ground_truth/committees-current.yaml")

# Build/update ground truth
print("=" * 60)
print("GROUND TRUTH EXTRACTION")
print("=" * 60)

df_gt = build_ground_truth(
    html_dir=GT_HTML_DIR,
    pew_path=GT_PEW_PATH,
    output_path=GT_OUTPUT_PATH,
    committee_yaml_path=GT_COMMITTEE_YAML_PATH,
    committees_yaml_path=GT_COMMITTEES_YAML_PATH,
    resume=True  # Skip already-processed senators; continue from checkpoint
)

GROUND TRUTH EXTRACTION
📂 Reading senator HTML files from ../external_data/senate_html...
   Found: 100 senators
   To process: 100 senators



Scraping senators:   0%|          | 0/100 [00:00<?, ?it/s]


🔗 Merging Pew religion data...

✓ Ground truth saved to: ../external_data/ground_truth/senate_ground_truth.csv
  Total senators: 100
  Columns: name, full_name, birthdate, gender, race_ethnicity, committee_roles, religion



### Ground Truth Validation and Loading

Validate and load ground truth CSV generated by wiki/ballotpedia scraping phase.
Ground truth includes: name, birthdate, gender, religion (Pew-sourced), committee_roles.
This is critical input for the evaluation phase that compares LLM predictions vs ground truth.

In [ ]:


OUTPUT_PATH = Path("../external_data/ground_truth/senate_ground_truth.csv")

if not OUTPUT_PATH.exists():
    print(f"⚠ Ground truth file not found: {OUTPUT_PATH} - evaluation will not work without it.")
    df_gt = None
else:
    df_gt = pd.read_csv(OUTPUT_PATH)
    print("Ground Truth Summary:")
    print(f"  Total senators: {len(df_gt)}")
    print(f"  Fields available: {len(df_gt.columns)}")
    print(f"  Empty fields per senator (avg): {df_gt.isna().sum().mean():.1f}")
    
    # Show field coverage
    print("\nField Coverage (Ground Truth):")
    for col in ["name", "birthdate", "gender", "religion", "committee_roles"]:
        if col in df_gt.columns:
            covered = df_gt[col].notna().sum()
            pct = 100 * covered / len(df_gt)
            print(f"  {col:20s}: {covered:3d}/{len(df_gt)} ({pct:5.1f}%)")
    
    print(f"\n✓ Ground truth ready for evaluation")

### Structured Evaluation and Scoring
Compare LLM predictions vs ground truth using modules from evaluation_suite.

Key features (from modules.evaluation_suite and modules.evaluator):
- Automatic result caching (evaluate_all_styles checks cache before recomputing)
- Stratified metrics: accuracy, ROUGE-1, BERT score, hierarchical religion match
- Component-level breakdown for education (degree/institution/year matching)
- Supports fuzzy name matching fallback for senator_id mismatches

Runtime: 30-90 seconds (much faster on second run due to caching)

In [ ]:
from modules import (
    load_and_merge_results,
    evaluate_all_styles,
    print_evaluation_summary,
)

In [ ]:
# Load and merge all predictions with ground truth
print("Loading predictions and ground truth...")
merged_data = load_and_merge_results(
    pred_path=OUTPUT_DIR / "task1_pii.csv",
    gt_path=Path("../external_data/ground_truth/senate_ground_truth.csv"),
    merge_method="exact"
)

In [ ]:
# Compute evaluation metrics with caching (Second run on same data will load cached results instead of recomputing)
print("\nComputing evaluation metrics...")
eval_results = evaluate_all_styles(
    merged_by_style=merged_data['merged_by_style'],
    output_dir=OUTPUT_DIR,
    overwrite=True  # Set to True to recompute even if cache exists
)

In [ ]:
# Print formatted summary
print_evaluation_summary(eval_results)

## Prompt Engineering Ablation Study

Rigorous comparison of all three prompt styles on a **fixed held-out subset of 25 senators**.
Each senator is extracted independently using direct, pseudocode, and ICL prompts.

**Purpose:** Quantify which fields benefit most from structured prompts (e.g., pseudocode) vs. demonstrations (e.g., ICL).
This replicates Liu et al.'s ablation methodology (Section 4.2, Table 13).

**Reproducibility:**
- Fixed random seed (42) ensures same 25 senators across runs
- Separate results file (`ablation_results.json`) keeps ablation independent from main pipeline
- Resume-safe — skips already-completed combinations
- 3-second rate limit between API calls to respect quota

**Expected outcome:** ICL shows largest gains on education, committee roles, and religious affiliation (Liu et al. reports ~15-25% improvement)

In [ ]:
# ABLATION STUDY EXECUTION
# Test all three prompt styles (direct, pseudocode, ICL) on the fixed ablation subset.
# Uses modules.api.call_groq() and modules.html_processing.extract_readable_text().
#
# Resume-safe implementation: 
#   - Loads existing ablation_results.json if present
#   - Tracks which (senator_id, style) combinations are already complete
#   - Continues from where it left off if interrupted
#
# Rate limit protection: 3-second delay between API calls

from modules import (
    TASK1_DIRECT, TASK1_PSEUDOCODE, TASK1_ICL,
    extract_readable_text, call_groq
)

# Define all prompt styles for ablation (same templates as main pipeline)
ABLATION_STYLES = {
    "direct": TASK1_DIRECT,
    "pseudocode": TASK1_PSEUDOCODE,
    "icl": TASK1_ICL
}

# Load or initialize ablation results from disk
ablation_path = OUTPUT_DIR / "ablation_results.json"

if ablation_path.exists():
    with open(ablation_path) as f:
        ablation_results = json.load(f)
    # Track which (senator_id, style) combos are already completed
    completed = set()
    for sid, styles_dict in ablation_results.items():
        for style in styles_dict.keys():
            completed.add((sid, style))
    print(f"✓ Resuming — {len(completed)} senator-style combinations already processed")
else:
    ablation_results = {}
    completed = set()

# Compute remaining tasks: each senator needs extraction with all 3 styles
ablation_tasks = [(sid, style) for sid in ablation_senators for style in ABLATION_STYLES.keys()]
remaining_tasks = [t for t in ablation_tasks if t not in completed]

print(f"Remaining ablation tasks: {len(remaining_tasks)}/{len(ablation_tasks)}")
print()

# Main ablation loop: extract each senator with each prompt style
for senator_id, style_name in tqdm(remaining_tasks, desc="Ablation loop"):
    # Load HTML and extract clean text (same preprocessing as main pipeline)
    html_file = [f for f in html_files if f.stem == senator_id]
    if not html_file:
        continue
    
    # Read and clean HTML
    html = html_file[0].read_text(encoding="utf-8", errors="ignore")
    text = extract_readable_text(html)
    
    # Extract with current style using Groq API (via modules.api.call_groq)
    prompt = ABLATION_STYLES[style_name]
    result = call_groq(prompt, text)
    
    # Store result in nested dict: {senator_id: {style_name: result, ...}, ...}
    if senator_id not in ablation_results:
        ablation_results[senator_id] = {}
    
    ablation_results[senator_id][style_name] = result
    
    # Save incrementally (critical for resume safety and data preservation)
    with open(ablation_path, "w") as f:
        json.dump(ablation_results, f, indent=2)
    
    # Rate limit protection — 3 seconds between API calls to avoid quota errors
    time.sleep(3)

print(f"\n✓ Ablation complete. Results saved to: ablation_results.json")

In [ ]:
# ABLATION COVERAGE COMPARISON
# Analyze field coverage per prompt style for the fixed ablation subset.
# This shows which fields benefit from each prompt engineering approach.
#
# Display table: rows = fields, columns = prompt styles (direct, pseudocode, ICL)
# Then calculate average coverage per style for high-level comparison.

print("\n" + "=" * 80)
print(" PROMPT ABLATION — FIELD COVERAGE COMPARISON")
print("=" * 80)
print(f"\nAblation subset size: {len(ablation_senators)} senators")
print(f"Ablation styles: {', '.join(ABLATION_STYLES.keys())}")
print()

# Calculate coverage for each field by prompt style
coverage_by_style = {}
for style in ABLATION_STYLES.keys():
    style_data = df_ablation[df_ablation["prompt_style"] == style].copy()

    error_mask = style_data.get("extraction_error", pd.Series(dtype=str)).notna()
    valid_data = style_data[~error_mask]
    n_errors = int(error_mask.sum())
    n_valid = len(valid_data)

    if n_valid == 0:
        print(f"  ⚠ {style}: all {n_errors} results are errors — delete ablation_results.json and rerun")
        continue

    coverage_by_style[style] = {}
    for field in T1_FIELDS_ABLATION:
        filled = int(valid_data[field].notna().sum())
        pct = (filled / n_valid * 100)
        coverage_by_style[style][field] = {"filled": filled, "total": n_valid, "pct": pct}

if not coverage_by_style:
    print("⚠ No valid results to display.")
else:
    # Print coverage table
    print(f"{'Field':<35} | {'DIRECT':^15} | {'PSEUDOCODE':^15} | {'ICL':^15}")
    print("-" * 80)
    for field in T1_FIELDS_ABLATION:
        row_str = f"{field:<35} | "
        for style in ["direct", "pseudocode", "icl"]:
            if style in coverage_by_style:
                c = coverage_by_style[style][field]
                row_str += f"{c['filled']}/{c['total']} ({c['pct']:5.1f}%) | "
            else:
                row_str += f"{'N/A':^15} | "
        print(row_str)
    print("=" * 80)

    # Show average coverage per style
    print("\n OVERALL COVERAGE BY PROMPT STYLE:")
    for style in ["direct", "pseudocode", "icl"]:
        if style in coverage_by_style:
            avg_cov = sum(coverage_by_style[style][f]["pct"] for f in T1_FIELDS_ABLATION) / len(T1_FIELDS_ABLATION)
            print(f"  {style:<15}: {avg_cov:6.1f}% (avg across all fields)")
        else:
            print(f"  {style:<15}: N/A (all errors)")

    print("\n KEY OBSERVATIONS:")
    print("  • Compare direct vs pseudocode: Liu et al. find pseudocode slightly better for structured fields")
    print("  • Compare direct/pseudocode vs ICL: Liu et al. find ICL best for occupation-type fields")
    print("    (committee_roles, education, religious_affiliation)")
    print("=" * 80 + "\n")

## Appendix: Religion Taxonomy & Hierarchical Matching

To give partial credit for related religions (e.g., Methodist → Christian),
we use a hierarchical taxonomy. Exact matches score 1.0, parent-child matches
(e.g., Methodist vs Christian) score 0.7, and unrelated religions score 0.0.

This better reflects partial correctness: extracting "Christian Methodist" when
GT is "Methodist" is partially correct even if not exact.

**Scoring Details:**
- **1.0** — Exact match (Methodist = Methodist)
- **0.7** — Same religious category but different names (Methodist vs Christian, Baptist vs Protestant)
- **0.0** — Different religions (Methodist vs Catholic, Christian vs Muslim)
- **NaN** — Either value missing (not scored)

This approach is implemented in `modules.evaluator.religion_match_score()` and uses the `RELIGION_HIERARCHY` dictionary from `modules.config_unified`.